In [12]:
import tensorflow as tf

# Configure TensorFlow to manage GPU memory usage better
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Set TensorFlow to only allocate memory as needed
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))



AttributeError: module 'tensorflow' has no attribute 'config'

In [13]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("GPU device: ", tf.test.gpu_device_name())
print("Is built with CUDA: ", tf.test.is_built_with_cuda())


AttributeError: module 'tensorflow' has no attribute '__version__'

In [8]:
import tensorflow as tf

In [9]:
print(tf.config.list_physical_devices('GPU'))  # Should list your GPU(s)

AttributeError: module 'tensorflow' has no attribute 'config'

In [2]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [3]:

from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from transformers import BertTokenizer, TFBertModel

import numpy as np


In [4]:
import keras
print(keras.__version__)


3.3.2


In [2]:
print(tf.config.list_physical_devices('GPU'))

[]


In [6]:
from tensorflow.python.client import device_lib

device_lib.list_local_devices()

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 15485990231451984207
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 6272581632
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 10820708290090600472
 physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 2070 with Max-Q Design, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [7]:
tf.config.list_physical_devices(
    device_type=None
)

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [8]:
#Read data
data = pd.read_excel('Complaints_data/Complaints_2020_2023.xlsx')
data.head()

,Complaint ID,NHTSA Refno,Manufacturer_name,MAKETXT,MODELTXT,YEAR,CRASH,FAILDATE,FIRE,INJURED,...,RESTRAINT_TYPE,DEALER_NAME,DEALER_TEL,DEALER_CITY,DEALER_STATE,DEALER_ZIP,PROD_TYPE,REPAIRED_YN,MEDICAL_ATTN,VEHICLE_TOWED
0,1633301,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
1,1633302,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
2,1633303,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018.0,N,20191221,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
3,1633304,11292385,Ford Motor Company,FORD,EXPLORER,2020.0,N,20191226,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
4,1633305,11292386,"General Motors, LLC",CHEVROLET,VOLT,2017.0,N,20190712,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254818 entries, 0 to 254817
Data columns (total 49 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Complaint ID       254818 non-null  int64  
 1   NHTSA Refno        254818 non-null  int64  
 2   Manufacturer_name  254811 non-null  object 
 3   MAKETXT            254811 non-null  object 
 4   MODELTXT           254811 non-null  object 
 5   YEAR               254811 non-null  float64
 6   CRASH              254818 non-null  object 
 7   FAILDATE           254818 non-null  int64  
 8   FIRE               254818 non-null  object 
 9   INJURED            254818 non-null  int64  
 10  DEATHS             254818 non-null  int64  
 11  COMPDESC           254812 non-null  object 
 12  CITY               254805 non-null  object 
 13  STATE              254818 non-null  object 
 14  VIN                249559 non-null  object 
 15  DATEA              254818 non-null  int64  
 16  LD

In [10]:
data['PROD_TYPE'].value_counts()

PROD_TYPE
V    251804
T      1722
E       874
C       411
Name: count, dtype: int64

In [11]:

# Initialize tokenizer and model from the 'transformers' library
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertModel.from_pretrained('bert-base-uncased')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [12]:

data['combined_text'] = data['COMPDESC'] + ' ' + data['CDESCR']
data['combined_text'].fillna('No Description', inplace=True)  # Replace NaNs with a placeholder string
data['combined_text'] = data['combined_text'].astype(str)


C:\Users\golla\AppData\Local\Temp\ipykernel_29784\607740989.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['combined_text'].fillna('No Description', inplace=True)  # Replace NaNs with a placeholder string


In [14]:


def get_bert_embeddings(texts, batch_size=32):
    model = TFBertModel.from_pretrained('bert-base-uncased')
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, max_length=512, truncation=True, padding='max_length', return_tensors='tf')
        outputs = model(inputs)
        embeddings.append(outputs.last_hidden_state[:, 0, :].numpy())
    
    return np.vstack(embeddings)

# tf.debugging.set_log_device_placement(True)

# # Specify the device context using the device name
# with tf.device('/GPU:0'): 

# Now call the function with the modified batch processing
embeddings = get_bert_embeddings(data['combined_text'].tolist(), batch_size=16)
print(f"Generated embeddings shape: {embeddings.shape}")


Physical devices cannot be modified after being initialized


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Generated embeddings shape: (254818, 768)


In [14]:
import numpy as np

# np.save('Complaints_data/embeddings.npy', embeddings)

embeddings = np.load('Complaints_data/embeddings.npy')


In [15]:
embeddings

array([[-0.38430837, -0.23753202,  0.61271167, ..., -0.71239126,
         0.29472786, -0.0209882 ],
       [-0.3785464 , -0.22604342,  0.6458467 , ..., -0.7398345 ,
         0.25720048,  0.00528393],
       [-0.40104452, -0.20896852,  0.63251615, ..., -0.73660314,
         0.3080391 , -0.00837477],
       ...,
       [-0.01673339, -0.33941275,  0.25943214, ..., -0.08848643,
         0.19260144,  0.13664971],
       [ 0.02110618, -0.2842619 ,  0.21654168, ..., -0.11202075,
         0.33860815,  0.08656389],
       [-0.36266947, -0.4869434 ,  0.15762408, ..., -0.03998525,
         0.01976589,  0.36142567]], dtype=float32)

In [19]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def optimal_clusters(data, max_k=10):
    scores = []
    for k in range(2, max_k+1):
        kmeans = KMeans(n_clusters=k, random_state=42).fit(data)
        score = silhouette_score(data, kmeans.labels_)
        scores.append(score)
    optimal_k = scores.index(max(scores)) + 2  # Adjusting because range starts at 2
    return optimal_k

# Calculate the optimal number of clusters
optimal_k = optimal_clusters(embeddings)
print(f"Optimal number of clusters: {optimal_k}")

# Perform KMeans clustering with the optimal number of clusters
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)
data['Cluster'] = cluster_labels


KeyboardInterrupt: 

In [ ]:
from collections import Counter

# Assume each cluster can be dynamically labeled based on top terms
def label_clusters(df, vectorizer, cluster_col='Cluster', text_col='combined_text', n_terms=5):
    cluster_labels_mapping = {}
    for cluster in df[cluster_col].unique():
        cluster_texts = df[df[cluster_col] == cluster][text_col]
        tfidf_matrix = vectorizer.transform(cluster_texts)
        sum_words = tfidf_matrix.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
        sorted_words = sorted(words_freq, key=lambda x: x[1], reverse=True)
        top_words = [word for word, freq in sorted_words[:n_terms]]
        # Labeling logic based on domain knowledge
        if 'battery' in top_words or 'circuit' in top_words:
            label = "Electrical Systems"
        elif 'engine' in top_words or 'transmission' in top_words:
            label = "Mechanical Systems"
        # Add more conditions as needed
        else:
            label = "Other"
        cluster_labels_mapping[cluster] = label
    return cluster_labels_mapping

# Get cluster labels
cluster_label_mapping = label_clusters(data, vectorizer)
data['Cluster_Label'] = data['Cluster'].map(cluster_label_mapping)


In [16]:
kmeans = KMeans(n_clusters=6, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)

# Add cluster labels to the sampled data
#sample_data = data.to_frame()
data['Cluster'] = cluster_labels


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Text vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(data['combined_text'])

# Function to get top keywords for each cluster
def get_top_keywords(X, clusters, labels, n_terms):
    df = pd.DataFrame(X.toarray(), columns=labels)
    df['cluster'] = clusters
    keywords = {}
    for i in range(6):
        words = df[df.cluster == i].mean().nlargest(n_terms).index.tolist()
        keywords[i] = words
    return keywords

# Get and print top keywords
keywords = get_top_keywords(X, data['Cluster'], vectorizer.get_feature_names_out(), 25)
for cluster_name, terms in keywords.items():
    print(f"Cluster {cluster_name}: {terms}")

Cluster 0: ['vehicle', 'engine', 'steering', 'car', 'power', 'electrical', 'control', 'driving', 'brakes', 'issue', 'fuel', 'light', 'brake', 'rear', 'service', 'unknown', 'air', 'problem', 'passenger', 'driver', 'door', 'recall', 'transmission', 'miles', 'causing']
Cluster 1: ['cluster', 'car', 'engine', 'unknown', 'tr', 'steering', 'power', 'driving', 'vehicle', 'air', 'light', 'electrical', 'control', 'train', 'bags', 'fuel', 'brakes', 'service', 'transmission', 'structure', 'miles', 'body', 'recall', 'windshield', 'highway']
Cluster 2: ['cluster', 'car', 'vehicle', 'issue', 'recall', 'engine', 'problem', 'dealership', 'safety', 'time', 'service', 'driving', 'told', 'miles', 'steering', 'light', 'did', 'brake', 'said', 'dealer', 'drive', 'power', 'oil', 'issues', 'control']
Cluster 3: ['cluster', 'car', 'vehicle', 'engine', 'steering', 'driving', 'issue', 'problem', 'power', 'control', 'light', 'miles', 'truck', 'time', 'recall', 'brakes', 'stop', 'times', 'highway', 'road', 'servic

In [18]:
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming you have your data as a pandas DataFrame named 'data'

# Text pre-processing (replace with your pre-processing steps if needed)
def preprocess_text(text):
  # Lowercase, remove stopwords, stemming/lemmatization (optional)
  text = text.lower()
  # Add your stopword removal and other cleaning steps here
  return text

data['combined_text'] = data['combined_text'].apply(preprocess_text)  # Apply pre-processing

# Feature extraction
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(data['combined_text'])

# Function to get cluster labels and features
def get_cluster_labels_and_features(data):
  clusters = data['Cluster'].tolist()
  features = vectorizer.get_feature_names_out()
  return clusters, features

# Get cluster labels and features
clusters, features = get_cluster_labels_and_features(data)

# Find unique words for each cluster (considering all documents)
unique_words = defaultdict(list)
for i, doc in enumerate(X.toarray()):
  word_indices = doc.nonzero()[0]  # Get indices of non-zero elements (words) in the document
  words = [features[idx] for idx in word_indices]  # Get word names based on indices
  for word in words:
    # Check if the word is not present in any other cluster's word list (excluding current cluster)
    if word not in [inner_cluster for inner_cluster in clusters if inner_cluster != clusters[i]]:
      unique_words[clusters[i]].append(word)

# Print the unique words for each cluster
for cluster_id, words in unique_words.items():
  print(f"Cluster {cluster_id}: {words}")


KeyboardInterrupt: 

In [ ]:
# Map clusters to predefined labels based on domain knowledge
cluster_labels_mapping = {
    0: "Mechanical Systems",   # Example, assuming keywords like 'battery', 'circuit'
    1: "Electrical Systems",   # Example, assuming keywords like 'engine', 'transmission'
    2: "Safety Features",      # Example, assuming keywords like 'airbag', 'seatbelt'
    3: "Interior Comfort",     # Example, assuming keywords like 'HVAC', 'seat', 'noise'
    4: "Software/Navigation",      # Example, assuming keywords like 'paint', 'door'
    5: "Exterior Issues"   # Example, assuming keywords like 'infotainment', 'gps'
}

data['Cluster_Label'] = data['Cluster'].map(cluster_labels_mapping)
print(data[['combined_text', 'Cluster_Label']].sample(5))

In [ ]:
X_input_ids = np.zeros((len(data), 512))
X_attn_masks = np.zeros((len(data), 512))

X_input_ids.shape

In [ ]:
X_attn_masks.shape

In [ ]:
#tokenize data
def generate_training_data(data, ids, masks, tokenizer):
  for i, text in tqdm(enumerate(data['CDESCR'])):
    tokenized_text = tokenizer.encode_plus(text, max_length = 512, truncation = True, padding = 'max_length', add_special_tokens= True)
    ids[i, :] = tokenized_text.input_ids
    masks[i, :] = tokenized_text.attention_mask
  return ids, masks

In [ ]:
tf.debugging.set_log_device_placement(True)

# Specify the device context using the device name
with tf.device('/GPU:0'):  # '/GPU:0' is the device name for the first GPU.
    X_input_ids, X_attn_masks = generate_training_data(data, X_input_ids, X_attn_masks, tokenizer )


In [ ]:
X_input_ids, X_attn_masks = generate_training_data(data, X_input_ids, X_attn_masks, tokenizer )


In [ ]:
X_attn_masks

In [ ]:
labels = np.zeros((len(data), 10))
labels


In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)

# Add cluster labels to the sampled data
sample_data = sample_data.to_frame()
sample_data['Cluster'] = cluster_labels
